# Final Project - Hanoi Noon Weather Forecasting with Machine Learning

## Academic Colab Notebook

This notebook is written in the same style as `notebook_labwork_2_final.ipynb`: it contains markdown explanations, helper functions, model training cells, evaluation tables, and plots directly inside the notebook.

The project predicts Hanoi weather at **12:00 noon** using Open-Meteo historical data.

### Main tasks

1. Next-day temperature regression.
2. Next-day rain classification.
3. 7-day direct multi-horizon temperature and rain forecasting.
4. Hyperparameter tuning with `RandomizedSearchCV` and `TimeSeriesSplit`.
5. Rain decision threshold tuning.
6. Generation of tables, figures, and an academic Markdown report.

### Required input file

Upload one of the following files when running in Colab:

- `open_meteo_hanoi.csv`
- `open-meteo-21.05N105.81E18m - open-meteo-21.05N105.81E18m.csv`


## 1. Install and import libraries


In [ ]:
!pip -q install xgboost joblib tabulate

import os
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
)
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBRegressor, XGBClassifier
except Exception as exc:
    XGBRegressor = None
    XGBClassifier = None
    print("XGBoost import failed:", exc)

pd.set_option("display.max_columns", 120)
RANDOM_STATE = 42
ROOT = Path(".")
REPORTS_DIR = ROOT / "reports_notebook"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
for d in [REPORTS_DIR, FIGURES_DIR, TABLES_DIR, ROOT / "data" / "raw"]:
    d.mkdir(parents=True, exist_ok=True)
print("Libraries imported successfully.")


## 2. Helper functions

The following functions load data, preprocess Open-Meteo CSV files, create supervised learning features, evaluate models, and save results.


In [ ]:
def find_file(possible_names):
    search_roots = [".", "/content", "/content/drive/MyDrive", "/kaggle/input"]
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirname, _, filenames in os.walk(root):
            for filename in filenames:
                if filename in possible_names:
                    return os.path.join(dirname, filename)
    return None


def upload_data_if_needed():
    possible = [
        "open_meteo_hanoi.csv",
        "open-meteo-21.05N105.81E18m - open-meteo-21.05N105.81E18m.csv",
    ]
    path = find_file(possible)
    if path is not None:
        print("Found dataset:", path)
        return Path(path)
    try:
        from google.colab import files
        print("Dataset not found. Please upload Open-Meteo CSV file.")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file uploaded.")
        name = next(iter(uploaded.keys()))
        target = ROOT / "data" / "raw" / "open_meteo_hanoi.csv"
        target.write_bytes(uploaded[name])
        print("Saved uploaded dataset to", target)
        return target
    except ModuleNotFoundError:
        raise FileNotFoundError("Dataset not found. Put open_meteo_hanoi.csv in the current directory or data/raw/.")


def load_open_meteo_csv(path):
    # Open-Meteo CSV may contain metadata lines before the actual header.
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    header_idx = 0
    for i, line in enumerate(lines):
        if line.lower().startswith("time"):
            header_idx = i
            break
    df = pd.read_csv(path, skiprows=header_idx)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df


def standardize_columns(df):
    rename = {}
    for col in df.columns:
        low = col.lower()
        if "temperature_2m" in low or low == "temperature": rename[col] = "temperature"
        elif "relative_humidity_2m" in low or low == "humidity": rename[col] = "humidity"
        elif low == "precipitation" or "precipitation" in low: rename[col] = "precipitation"
        elif "cloud_cover" in low: rename[col] = "cloud_cover"
        elif "pressure_msl" in low: rename[col] = "pressure_msl"
        elif "wind_speed_10m" in low: rename[col] = "wind_speed_10m"
        elif "shortwave_radiation" in low: rename[col] = "shortwave_radiation"
        elif low == "time": rename[col] = "time"
    return df.rename(columns=rename)


def preprocess_weather(df):
    df = standardize_columns(df.copy())
    if "time" not in df.columns:
        raise ValueError("The dataset must contain a time column.")
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time").drop_duplicates("time")
    df = df[df["time"].dt.hour == 12].copy()
    required = ["temperature", "humidity", "precipitation", "cloud_cover", "pressure_msl", "wind_speed_10m", "shortwave_radiation"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    keep = ["time"] + required
    df = df[keep].copy()
    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())
    return df.reset_index(drop=True)


def weather_level(temp):
    if temp < 22: return "cool"
    if temp <= 30: return "normal"
    return "hot"


def create_features(noon_df):
    df = noon_df.copy().sort_values("time").reset_index(drop=True)
    df["month"] = df["time"].dt.month
    df["day_of_year"] = df["time"].dt.dayofyear
    df["season"] = ((df["month"] % 12) // 3 + 1).astype(int)
    for col in ["temperature", "humidity", "precipitation"]:
        for lag in [1, 3, 7]:
            df[f"{col.split('_')[0]}_lag{lag}"] = df[col].shift(lag)
    df["temp_roll_mean_3"] = df["temperature"].shift(1).rolling(3).mean()
    df["temp_roll_mean_7"] = df["temperature"].shift(1).rolling(7).mean()
    df["humidity_roll_mean_3"] = df["humidity"].shift(1).rolling(3).mean()
    df["humidity_roll_mean_7"] = df["humidity"].shift(1).rolling(7).mean()
    df["rain_roll_sum_3"] = df["precipitation"].shift(1).rolling(3).sum()
    df["rain_roll_sum_7"] = df["precipitation"].shift(1).rolling(7).sum()
    df["target_temperature"] = df["temperature"].shift(-1)
    df["target_precipitation"] = df["precipitation"].shift(-1)
    df["target_rain"] = (df["target_precipitation"] > 0).astype(float)
    df["target_weather_level"] = df["target_temperature"].apply(lambda x: weather_level(x) if pd.notna(x) else np.nan)
    return df.dropna().reset_index(drop=True)


def chronological_split(df):
    train = df[df["time"].dt.year <= 2023].copy()
    val = df[df["time"].dt.year == 2024].copy()
    test = df[df["time"].dt.year == 2025].copy()
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def select_feature_columns(df, include_shortwave=True):
    blocked = {"time", "target_temperature", "target_precipitation", "target_rain", "target_weather_level"}
    if not include_shortwave:
        blocked.add("shortwave_radiation")
    return [c for c in df.select_dtypes(include="number").columns if c not in blocked]


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def save_table(df, name):
    path = TABLES_DIR / name
    df.to_csv(path, index=False)
    print("Saved", path)
    return path


def display_and_save(fig_name):
    path = FIGURES_DIR / fig_name
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()
    print("Saved", path)


## 3. Load and inspect dataset


In [ ]:
csv_path = upload_data_if_needed()
raw_df = load_open_meteo_csv(csv_path)
noon_df = preprocess_weather(raw_df)
feature_df = create_features(noon_df)
train_df, val_df, test_df = chronological_split(feature_df)

print("Raw hourly records:", len(raw_df))
print("Noon records:", len(noon_df))
print("Supervised samples:", len(feature_df))
print("Date range:", noon_df["time"].min(), "to", noon_df["time"].max())
print("Train / Validation / Test:", len(train_df), len(val_df), len(test_df))
print("Next-day rain rate:", feature_df["target_rain"].mean().round(3))
noon_df.head()


## 4. Descriptive statistics and EDA


In [ ]:
stats = noon_df[["temperature", "humidity", "precipitation", "cloud_cover", "pressure_msl", "wind_speed_10m", "shortwave_radiation"]].describe().round(3)
save_table(stats.reset_index(), "descriptive_statistics.csv")
stats


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(noon_df["time"], noon_df["temperature"], linewidth=1)
plt.title("Noon temperature trend in Hanoi")
plt.xlabel("Time")
plt.ylabel("Temperature (degC)")
display_and_save("temperature_trend.png")

plt.figure(figsize=(7,4))
plt.hist(noon_df["temperature"], bins=30, edgecolor="black")
plt.title("Noon temperature distribution")
plt.xlabel("Temperature (degC)")
plt.ylabel("Frequency")
display_and_save("temperature_histogram.png")

corr_cols = ["temperature", "humidity", "precipitation", "cloud_cover", "pressure_msl", "wind_speed_10m", "shortwave_radiation"]
corr = noon_df[corr_cols].corr()
plt.figure(figsize=(8,6))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
plt.yticks(range(len(corr_cols)), corr_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation heatmap")
display_and_save("correlation_heatmap.png")


In [ ]:
from sklearn.decomposition import PCA

feature_cols_for_pca = select_feature_columns(feature_df, include_shortwave=True)
X_scaled = StandardScaler().fit_transform(feature_df[feature_cols_for_pca])
pca = PCA(n_components=2, random_state=RANDOM_STATE)
components = pca.fit_transform(X_scaled)
plt.figure(figsize=(7,5))
plt.scatter(components[:,0], components[:,1], c=feature_df["target_rain"], cmap="coolwarm", alpha=0.65, s=18)
plt.title("PCA visualization colored by next-day rain label")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Rain label")
display_and_save("pca_visualization.png")
print("Explained variance ratio:", pca.explained_variance_ratio_)


## 5. Baselines


In [ ]:
def regression_baseline(df):
    pred = df["temperature"].to_numpy()
    y = df["target_temperature"].to_numpy()
    return {"model": "PersistenceBaseline", "mae": mean_absolute_error(y, pred), "rmse": rmse(y, pred), "r2": r2_score(y, pred)}


def rain_baseline(train_df, eval_df):
    majority = int(train_df["target_rain"].mode().iloc[0])
    y = eval_df["target_rain"].astype(int).to_numpy()
    pred = np.full_like(y, majority)
    return {
        "model": "MajorityClassBaseline",
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
    }

baseline_rows = []
for split_name, df in [("validation", val_df), ("test", test_df)]:
    baseline_rows.append({"split": split_name, "task": "temperature_regression", **regression_baseline(df)})
    baseline_rows.append({"split": split_name, "task": "rain_classification", **rain_baseline(train_df, df)})
baseline_df = pd.DataFrame(baseline_rows)
save_table(baseline_df, "baseline_results.csv")
baseline_df


## 6. Next-day model training

The temperature task compares Linear Regression, KNN Regressor, and XGBoost Regressor.  
The rain task compares Logistic Regression, SVM, and XGBoost Classifier.


In [ ]:
def train_temperature_models(X_train, y_train, X_val, y_val, search_iter=4):
    models = {
        "LinearRegression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
        "KNNRegressor": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsRegressor())]),
    }
    searches = {
        "KNNRegressor": {"model__n_neighbors": randint(3, 26), "model__weights": ["uniform", "distance"]}
    }
    if XGBRegressor is not None:
        models["XGBoostRegressor"] = XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=1, verbosity=0)
        searches["XGBoostRegressor"] = {"n_estimators": randint(100, 501), "max_depth": randint(3, 11), "learning_rate": uniform(0.01, 0.29)}

    cv = TimeSeriesSplit(n_splits=5)
    rows, fitted = [], {}
    for name, model in models.items():
        start = time.perf_counter()
        if name in searches:
            model = RandomizedSearchCV(model, searches[name], n_iter=search_iter, cv=cv, scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE, n_jobs=1)
        model.fit(X_train, y_train)
        elapsed = time.perf_counter() - start
        pred = model.predict(X_val)
        rows.append({"model": name, "mae": mean_absolute_error(y_val, pred), "rmse": rmse(y_val, pred), "r2": r2_score(y_val, pred), "training_time": elapsed, "best_params": json.dumps(getattr(model, "best_params_", {}))})
        fitted[name] = model
    return pd.DataFrame(rows), fitted


def rain_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    return model.predict(X)


def train_rain_models(X_train, y_train, X_val, y_val, search_iter=4):
    y_train = y_train.astype(int)
    y_val = y_val.astype(int)
    models = {
        "LogisticRegression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))]),
        "SVM": Pipeline([("scaler", StandardScaler()), ("model", SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE))]),
    }
    searches = {
        "LogisticRegression": {"model__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
        "SVM": {"model__C": [0.1, 1.0, 10.0, 100.0], "model__kernel": ["linear", "rbf"]},
    }
    if XGBClassifier is not None:
        neg, pos = max(1, (y_train == 0).sum()), max(1, (y_train == 1).sum())
        models["XGBoostClassifier"] = XGBClassifier(objective="binary:logistic", eval_metric="logloss", scale_pos_weight=neg/pos, random_state=RANDOM_STATE, n_jobs=1, verbosity=0)
        searches["XGBoostClassifier"] = {"n_estimators": randint(100, 501), "max_depth": randint(3, 11), "learning_rate": uniform(0.01, 0.29)}

    cv = TimeSeriesSplit(n_splits=5)
    rows, fitted = [], {}
    for name, model in models.items():
        start = time.perf_counter()
        n_iter = min(search_iter, len(list(ParameterGrid(searches[name]))) if all(isinstance(v, list) for v in searches[name].values()) else search_iter)
        model = RandomizedSearchCV(model, searches[name], n_iter=n_iter, cv=cv, scoring="f1", random_state=RANDOM_STATE, n_jobs=1)
        model.fit(X_train, y_train)
        elapsed = time.perf_counter() - start
        pred = model.predict(X_val).astype(int)
        prob = rain_scores(model, X_val)
        rows.append({
            "model": name,
            "accuracy": accuracy_score(y_val, pred),
            "precision": precision_score(y_val, pred, zero_division=0),
            "recall": recall_score(y_val, pred, zero_division=0),
            "f1": f1_score(y_val, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_val, prob),
            "training_time": elapsed,
            "best_params": json.dumps(getattr(model, "best_params_", {})),
        })
        fitted[name] = model
    return pd.DataFrame(rows), fitted


In [ ]:
feature_cols = select_feature_columns(feature_df, include_shortwave=True)
X_train, X_val, X_test = train_df[feature_cols], val_df[feature_cols], test_df[feature_cols]
y_temp_train, y_temp_val, y_temp_test = train_df["target_temperature"], val_df["target_temperature"], test_df["target_temperature"]
y_rain_train, y_rain_val, y_rain_test = train_df["target_rain"], val_df["target_rain"], test_df["target_rain"]

temp_val_df, temp_models = train_temperature_models(X_train, y_temp_train, X_val, y_temp_val, search_iter=4)
rain_val_df, rain_models = train_rain_models(X_train, y_rain_train, X_val, y_rain_val, search_iter=4)

best_temp_model = temp_val_df.sort_values("rmse").iloc[0]["model"]
best_rain_model = rain_val_df.sort_values("f1", ascending=False).iloc[0]["model"]
print("Best validation temperature model:", best_temp_model)
print("Best validation rain model:", best_rain_model)

def evaluate_temp_test(models):
    rows = []
    for name, model in models.items():
        pred = model.predict(X_test)
        rows.append({"model": name, "mae": mean_absolute_error(y_temp_test, pred), "rmse": rmse(y_temp_test, pred), "r2": r2_score(y_temp_test, pred)})
    return pd.DataFrame(rows)


def evaluate_rain_test(models):
    rows = []
    y = y_rain_test.astype(int)
    for name, model in models.items():
        pred = model.predict(X_test).astype(int)
        prob = rain_scores(model, X_test)
        rows.append({"model": name, "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0), "roc_auc": roc_auc_score(y, prob)})
    return pd.DataFrame(rows)

temp_test_df = evaluate_temp_test(temp_models)
rain_test_df = evaluate_rain_test(rain_models)

save_table(temp_val_df, "validation_regression_results.csv")
save_table(rain_val_df, "validation_rain_classification_results.csv")
save_table(temp_test_df, "regression_results.csv")
save_table(rain_test_df, "rain_classification_results.csv")

display(temp_val_df)
display(temp_test_df)
display(rain_val_df)
display(rain_test_df)


## 7. Next-day visualizations and Linear Regression coefficients


In [ ]:
# Model comparison line charts
plt.figure(figsize=(10,5))
reg_long = temp_test_df.melt(id_vars="model", value_vars=["mae", "rmse", "r2"], var_name="metric", value_name="value")
for model in reg_long["model"].unique():
    part = reg_long[reg_long["model"] == model]
    plt.plot(part["metric"], part["value"], marker="o", label=model)
plt.title("Next-day temperature regression model comparison")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.grid(alpha=0.25)
plt.legend()
display_and_save("regression_model_comparison_line.png")

plt.figure(figsize=(11,5))
rain_long = rain_test_df.melt(id_vars="model", value_vars=["accuracy", "precision", "recall", "f1", "roc_auc"], var_name="metric", value_name="value")
for model in rain_long["model"].unique():
    part = rain_long[rain_long["model"] == model]
    plt.plot(part["metric"], part["value"], marker="o", label=model)
plt.title("Next-day rain classification model comparison")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend()
display_and_save("classification_model_comparison_line.png")


In [ ]:
best_temp_pred = temp_models[best_temp_model].predict(X_test)
best_rain_pred = rain_models[best_rain_model].predict(X_test).astype(int)

plt.figure(figsize=(12,5))
plt.plot(test_df["time"], y_temp_test, label="Actual", linewidth=1.2)
plt.plot(test_df["time"], best_temp_pred, label="Predicted", linewidth=1.2)
plt.title(f"Actual vs predicted next-day temperature on test set ({best_temp_model})")
plt.xlabel("Date")
plt.ylabel("Temperature (degC)")
plt.legend()
display_and_save("test_actual_vs_predicted_temperature.png")

residuals = y_temp_test - best_temp_pred
plt.figure(figsize=(10,4))
plt.scatter(best_temp_pred, residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.title("Temperature prediction residuals")
plt.xlabel("Predicted temperature")
plt.ylabel("Residual")
display_and_save("test_temperature_residuals.png")

cm = confusion_matrix(y_rain_test.astype(int), best_rain_pred)
plt.figure(figsize=(4,4))
plt.imshow(cm, cmap="Blues")
plt.title(f"Rain confusion matrix ({best_rain_model})")
plt.xticks([0,1], ["No rain", "Rain"])
plt.yticks([0,1], ["No rain", "Rain"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")
plt.xlabel("Predicted")
plt.ylabel("Actual")
display_and_save("test_confusion_matrix_rain.png")
print(classification_report(y_rain_test.astype(int), best_rain_pred, zero_division=0))


In [ ]:
# Linear Regression w and b interpretation
lin = temp_models.get("LinearRegression")
if lin is not None:
    coefs = lin.named_steps["model"].coef_
    intercept = lin.named_steps["model"].intercept_
    coef_df = pd.DataFrame({"feature": feature_cols, "coefficient_w": coefs})
    coef_df["absolute_coefficient"] = coef_df["coefficient_w"].abs()
    coef_df["direction"] = np.where(coef_df["coefficient_w"] >= 0, "positive", "negative")
    coef_df = coef_df.sort_values("absolute_coefficient", ascending=False)
    save_table(coef_df, "linear_regression_coefficients.csv")
    print("Linear Regression intercept b:", intercept)
    display(coef_df.head(15))
    plt.figure(figsize=(9,6))
    top = coef_df.head(12).sort_values("coefficient_w")
    plt.barh(top["feature"], top["coefficient_w"])
    plt.title("Top Linear Regression coefficients w")
    plt.xlabel("Coefficient value after feature scaling")
    display_and_save("linear_regression_coefficients.png")


## 8. 7-day direct multi-horizon forecasting functions

Each horizon has its own target and model pair. This avoids recursive error accumulation.


In [ ]:
def build_horizon_dataset(noon_df, base_feature_df, horizon):
    target = noon_df[["time", "temperature", "precipitation"]].copy().sort_values("time")
    target["horizon_temperature"] = target["temperature"].shift(-horizon)
    target["horizon_precipitation"] = target["precipitation"].shift(-horizon)
    target["horizon_rain"] = (target["horizon_precipitation"] > 0).astype(float)
    target = target[["time", "horizon_temperature", "horizon_precipitation", "horizon_rain"]]
    df = base_feature_df.drop(columns=["target_temperature", "target_precipitation", "target_rain", "target_weather_level"], errors="ignore").merge(target, on="time", how="inner")
    return df.dropna(subset=["horizon_temperature", "horizon_precipitation", "horizon_rain"]).reset_index(drop=True)


def tune_thresholds(y_true, probs):
    rows = []
    best = {"threshold": 0.5, "f1": -1}
    for threshold in np.round(np.arange(0.10, 0.91, 0.05), 2):
        pred = (probs >= threshold).astype(int)
        row = {
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
        }
        rows.append(row)
        if row["f1"] > best["f1"]:
            best = {"threshold": threshold, "f1": row["f1"]}
    return best, rows


def train_and_evaluate_7day(search_iter=4, output_name="seven_day_forecast_notebook"):
    out_dir = REPORTS_DIR / output_name
    fig_dir = out_dir / "figures"
    tab_dir = out_dir / "tables"
    fig_dir.mkdir(parents=True, exist_ok=True)
    tab_dir.mkdir(parents=True, exist_ok=True)

    selected_rows, metric_rows, threshold_rows, forecast_rows, test_pred_rows = [], [], [], [], []
    tuning_rows = []
    feature_cols_7 = select_feature_columns(feature_df, include_shortwave=True)

    for horizon in range(1, 8):
        hdf = build_horizon_dataset(noon_df, feature_df, horizon)
        tr, va, te = chronological_split(hdf)
        Xtr, Xva, Xte = tr[feature_cols_7], va[feature_cols_7], te[feature_cols_7]
        ytr_temp, yva_temp, yte_temp = tr["horizon_temperature"], va["horizon_temperature"], te["horizon_temperature"]
        ytr_rain, yva_rain, yte_rain = tr["horizon_rain"].astype(int), va["horizon_rain"].astype(int), te["horizon_rain"].astype(int)

        temp_val, temp_fit = train_temperature_models(Xtr, ytr_temp, Xva, yva_temp, search_iter=search_iter)
        rain_val, rain_fit = train_rain_models(Xtr, ytr_rain, Xva, yva_rain, search_iter=search_iter)

        temp_test = []
        for name, model in temp_fit.items():
            pred = model.predict(Xte)
            temp_test.append({"horizon_day": horizon, "split": "test", "task": "temperature_regression", "model": name, "mae": mean_absolute_error(yte_temp, pred), "rmse": rmse(yte_temp, pred), "r2": r2_score(yte_temp, pred), "best_params": json.dumps(getattr(model, "best_params_", {}))})
            metric_rows.append({"horizon_day": horizon, "split": "validation", "task": "temperature_regression", **temp_val[temp_val["model"] == name].iloc[0].to_dict()})
            metric_rows.append(temp_test[-1])

        rain_val_with_threshold = []
        rain_test = []
        for name, model in rain_fit.items():
            val_prob = rain_scores(model, Xva)
            best_thr, curve = tune_thresholds(yva_rain, val_prob)
            for r in curve:
                threshold_rows.append({"horizon_day": horizon, "model": name, **r})
            val_pred = (val_prob >= best_thr["threshold"]).astype(int)
            test_prob = rain_scores(model, Xte)
            test_pred = (test_prob >= best_thr["threshold"]).astype(int)
            rv = {"horizon_day": horizon, "split": "validation", "task": "rain_classification", "model": name, "threshold": best_thr["threshold"], "accuracy": accuracy_score(yva_rain, val_pred), "precision": precision_score(yva_rain, val_pred, zero_division=0), "recall": recall_score(yva_rain, val_pred, zero_division=0), "f1": f1_score(yva_rain, val_pred, zero_division=0), "best_params": json.dumps(getattr(model, "best_params_", {}))}
            rt = {"horizon_day": horizon, "split": "test", "task": "rain_classification", "model": name, "threshold": best_thr["threshold"], "accuracy": accuracy_score(yte_rain, test_pred), "precision": precision_score(yte_rain, test_pred, zero_division=0), "recall": recall_score(yte_rain, test_pred, zero_division=0), "f1": f1_score(yte_rain, test_pred, zero_division=0), "best_params": json.dumps(getattr(model, "best_params_", {}))}
            rain_val_with_threshold.append(rv)
            rain_test.append(rt)
            metric_rows.extend([rv, rt])

        temp_val_h = pd.DataFrame([r for r in metric_rows if r.get("horizon_day") == horizon and r.get("split") == "validation" and r.get("task") == "temperature_regression"])
        rain_val_h = pd.DataFrame(rain_val_with_threshold)
        best_temp = temp_val_h.sort_values("rmse").iloc[0]["model"]
        best_rain = rain_val_h.sort_values("f1", ascending=False).iloc[0]["model"]
        best_rain_threshold = rain_val_h[rain_val_h["model"] == best_rain]["threshold"].iloc[0]
        temp_test_h = pd.DataFrame(temp_test)
        rain_test_h = pd.DataFrame(rain_test)

        selected_rows.append({
            "horizon_day": horizon,
            "best_temperature_model": best_temp,
            "validation_rmse": temp_val_h[temp_val_h["model"] == best_temp]["rmse"].iloc[0],
            "test_rmse": temp_test_h[temp_test_h["model"] == best_temp]["rmse"].iloc[0],
            "best_rain_model": best_rain,
            "validation_f1": rain_val_h[rain_val_h["model"] == best_rain]["f1"].iloc[0],
            "test_f1": rain_test_h[rain_test_h["model"] == best_rain]["f1"].iloc[0],
            "rain_threshold": best_rain_threshold,
        })

        temp_pred = temp_fit[best_temp].predict(Xte)
        rain_prob = rain_scores(rain_fit[best_rain], Xte)
        rain_pred = (rain_prob >= best_rain_threshold).astype(int)
        for i, (_, row) in enumerate(te.reset_index(drop=True).iterrows()):
            test_pred_rows.append({"horizon_day": horizon, "time": row["time"], "temperature_model": best_temp, "rain_model": best_rain, "actual_temperature": row["horizon_temperature"], "predicted_temperature": temp_pred[i], "actual_rain": int(row["horizon_rain"]), "predicted_rain": int(rain_pred[i]), "rain_correct": int(row["horizon_rain"]) == int(rain_pred[i])})

        latest_features = feature_df.sort_values("time").iloc[[-1]][feature_cols_7]
        f_temp = float(temp_fit[best_temp].predict(latest_features)[0])
        f_prob = float(rain_scores(rain_fit[best_rain], latest_features)[0])
        f_rain = int(f_prob >= best_rain_threshold)
        forecast_rows.append({"horizon_day": horizon, "forecast_time": feature_df["time"].max() + pd.Timedelta(days=horizon), "temperature_model": best_temp, "rain_model": best_rain, "predicted_temperature": f_temp, "predicted_rain_probability": f_prob, "rain_threshold": best_rain_threshold, "predicted_rain_label": "rain" if f_rain else "no_rain", "predicted_weather_level": weather_level(f_temp)})

    selected_df = pd.DataFrame(selected_rows)
    metrics_df = pd.DataFrame(metric_rows)
    threshold_df = pd.DataFrame(threshold_rows)
    forecast_df = pd.DataFrame(forecast_rows)
    test_predictions_df = pd.DataFrame(test_pred_rows)
    selected_df.to_csv(tab_dir / "seven_day_selected_models.csv", index=False)
    metrics_df.to_csv(tab_dir / "seven_day_horizon_metrics.csv", index=False)
    threshold_df.to_csv(tab_dir / "seven_day_threshold_tuning.csv", index=False)
    forecast_df.to_csv(tab_dir / "seven_day_forecast_predictions.csv", index=False)
    test_predictions_df.to_csv(tab_dir / "seven_day_test_predictions.csv", index=False)

    # Line model comparison plots
    test_metrics = metrics_df[metrics_df["split"] == "test"]
    plt.figure(figsize=(10,5))
    for model in test_metrics[test_metrics["task"] == "temperature_regression"]["model"].unique():
        part = test_metrics[(test_metrics["task"] == "temperature_regression") & (test_metrics["model"] == model)]
        plt.plot(part["horizon_day"], part["rmse"], marker="o", label=model)
    plt.title(f"7-day temperature model comparison ({output_name})")
    plt.xlabel("Forecast horizon")
    plt.ylabel("Test RMSE")
    plt.legend(); plt.grid(alpha=0.25); plt.tight_layout(); plt.savefig(fig_dir / "seven_day_temperature_model_comparison.png", dpi=160); plt.show()

    plt.figure(figsize=(10,5))
    for model in test_metrics[test_metrics["task"] == "rain_classification"]["model"].unique():
        part = test_metrics[(test_metrics["task"] == "rain_classification") & (test_metrics["model"] == model)]
        plt.plot(part["horizon_day"], part["f1"], marker="o", label=model)
    plt.title(f"7-day rain model comparison ({output_name})")
    plt.xlabel("Forecast horizon")
    plt.ylabel("Test F1")
    plt.ylim(0, 1); plt.legend(); plt.grid(alpha=0.25); plt.tight_layout(); plt.savefig(fig_dir / "seven_day_rain_model_comparison.png", dpi=160); plt.show()

    # Threshold tuning plot for selected rain models
    fig, axes = plt.subplots(7, 1, figsize=(11, 22), sharex=True)
    for ax, (_, selected) in zip(axes, selected_df.iterrows()):
        h, model, thr = selected["horizon_day"], selected["best_rain_model"], selected["rain_threshold"]
        part = threshold_df[(threshold_df["horizon_day"] == h) & (threshold_df["model"] == model)]
        ax.plot(part["threshold"], part["f1"], marker="o", label=model)
        ax.axvline(thr, color="red", linestyle="--", label=f"best threshold={thr:.2f}")
        ax.set_title(f"Horizon {h}: threshold tuning")
        ax.set_ylabel("F1"); ax.set_ylim(0, 1); ax.legend(loc="lower right")
    axes[-1].set_xlabel("Decision threshold")
    plt.tight_layout(); plt.savefig(fig_dir / "rain_threshold_tuning_by_horizon.png", dpi=160); plt.show()

    return {"output_dir": out_dir, "selected": selected_df, "metrics": metrics_df, "threshold": threshold_df, "forecast": forecast_df, "test_predictions": test_predictions_df}


## 9. Run 7-day forecasting: n_iter=4 and n_iter=20


In [ ]:
seven_iter4 = train_and_evaluate_7day(search_iter=4, output_name="seven_day_forecast_enhanced_iter4")
print("Mean test RMSE n_iter=4:", seven_iter4["selected"]["test_rmse"].mean())
print("Mean test F1 n_iter=4:", seven_iter4["selected"]["test_f1"].mean())
display(seven_iter4["selected"])


In [ ]:
seven_iter20 = train_and_evaluate_7day(search_iter=20, output_name="seven_day_forecast_enhanced_iter20")
print("Mean test RMSE n_iter=20:", seven_iter20["selected"]["test_rmse"].mean())
print("Mean test F1 n_iter=20:", seven_iter20["selected"]["test_f1"].mean())
display(seven_iter20["selected"])
display(seven_iter20["forecast"])


## 10. Compare n_iter=4 vs n_iter=20


In [ ]:
comparison = seven_iter4["selected"][["horizon_day", "best_temperature_model", "test_rmse", "best_rain_model", "test_f1"]].rename(columns={"best_temperature_model":"iter4_temp_model", "test_rmse":"iter4_temp_rmse", "best_rain_model":"iter4_rain_model", "test_f1":"iter4_rain_f1"})
comparison = comparison.merge(
    seven_iter20["selected"][["horizon_day", "best_temperature_model", "test_rmse", "best_rain_model", "test_f1"]].rename(columns={"best_temperature_model":"iter20_temp_model", "test_rmse":"iter20_temp_rmse", "best_rain_model":"iter20_rain_model", "test_f1":"iter20_rain_f1"}),
    on="horizon_day"
)
comparison["rmse_delta_iter20_minus_iter4"] = comparison["iter20_temp_rmse"] - comparison["iter4_temp_rmse"]
comparison["f1_delta_iter20_minus_iter4"] = comparison["iter20_rain_f1"] - comparison["iter4_rain_f1"]
save_table(comparison, "seven_day_iter4_vs_iter20_comparison.csv")
display(comparison)

plt.figure(figsize=(10,5))
plt.plot(comparison["horizon_day"], comparison["iter4_temp_rmse"], marker="o", label="n_iter=4")
plt.plot(comparison["horizon_day"], comparison["iter20_temp_rmse"], marker="o", label="n_iter=20")
plt.title("Selected temperature model test RMSE: n_iter=4 vs n_iter=20")
plt.xlabel("Forecast horizon")
plt.ylabel("Test RMSE")
plt.legend(); plt.grid(alpha=0.25)
display_and_save("iter4_vs_iter20_temperature_rmse.png")

plt.figure(figsize=(10,5))
plt.plot(comparison["horizon_day"], comparison["iter4_rain_f1"], marker="o", label="n_iter=4")
plt.plot(comparison["horizon_day"], comparison["iter20_rain_f1"], marker="o", label="n_iter=20")
plt.title("Selected rain model test F1: n_iter=4 vs n_iter=20")
plt.xlabel("Forecast horizon")
plt.ylabel("Test F1")
plt.ylim(0, 1)
plt.legend(); plt.grid(alpha=0.25)
display_and_save("iter4_vs_iter20_rain_f1.png")


## 11. Additional plots required by the final academic report

This section adds the remaining artifacts so that the notebook output matches the final DOCX report: A/B shortwave experiment, 7-day forecast plots, full-test actual-vs-predicted plots, feature importance, and detailed hyperparameter tuning plots.


### 11.1 A/B experiment with and without shortwave radiation


In [ ]:
ab_rows = []
for include_shortwave, label in [(False, "A_without_shortwave_radiation"), (True, "B_with_shortwave_radiation")]:
    cols = select_feature_columns(feature_df, include_shortwave=include_shortwave)
    Xtr, Xva, Xte = train_df[cols], val_df[cols], test_df[cols]
    temp_val_ab, temp_models_ab = train_temperature_models(Xtr, y_temp_train, Xva, y_temp_val, search_iter=4)
    rain_val_ab, rain_models_ab = train_rain_models(Xtr, y_rain_train, Xva, y_rain_val, search_iter=4)

    temp_test_ab = []
    for name, model in temp_models_ab.items():
        pred = model.predict(Xte)
        temp_test_ab.append({"model": name, "rmse": rmse(y_temp_test, pred)})
    temp_test_ab = pd.DataFrame(temp_test_ab)

    rain_test_ab = []
    for name, model in rain_models_ab.items():
        pred = model.predict(Xte).astype(int)
        rain_test_ab.append({"model": name, "f1": f1_score(y_rain_test.astype(int), pred, zero_division=0)})
    rain_test_ab = pd.DataFrame(rain_test_ab)

    best_reg = temp_val_ab.sort_values("rmse").iloc[0]["model"]
    best_rain = rain_val_ab.sort_values("f1", ascending=False).iloc[0]["model"]
    ab_rows.append({
        "experiment": label,
        "include_shortwave_radiation": include_shortwave,
        "best_regression": best_reg,
        "validation_rmse": temp_val_ab[temp_val_ab["model"] == best_reg]["rmse"].iloc[0],
        "test_rmse": temp_test_ab[temp_test_ab["model"] == best_reg]["rmse"].iloc[0],
        "best_rain": best_rain,
        "validation_f1": rain_val_ab[rain_val_ab["model"] == best_rain]["f1"].iloc[0],
        "test_f1": rain_test_ab[rain_test_ab["model"] == best_rain]["f1"].iloc[0],
    })

ab_df = pd.DataFrame(ab_rows)
save_table(ab_df, "experiment_ab_comparison.csv")
display(ab_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ab_df["experiment"], ab_df["test_rmse"], marker="o")
axes[0].set_title("A/B test RMSE for temperature")
axes[0].set_ylabel("Test RMSE")
axes[0].tick_params(axis="x", rotation=15)
axes[1].plot(ab_df["experiment"], ab_df["test_f1"], marker="o", color="tab:green")
axes[1].set_title("A/B test F1 for rain")
axes[1].set_ylabel("Test F1")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=15)
plt.suptitle("A/B experiment: shortwave radiation feature")
display_and_save("experiment_ab_comparison.png")


### 11.2 Feature importance and model explanation plots


In [ ]:
# XGBoost Regressor feature importance
if "XGBoostRegressor" in temp_models:
    xgb_model = temp_models["XGBoostRegressor"]
    importances = getattr(xgb_model, "feature_importances_", None)
    if importances is not None:
        xgb_imp = pd.DataFrame({"feature": feature_cols, "importance": importances})
        xgb_imp = xgb_imp.sort_values("importance", ascending=False)
        save_table(xgb_imp, "xgboost_regressor_feature_importance.csv")
        display(xgb_imp.head(15))
        plt.figure(figsize=(9, 6))
        top = xgb_imp.head(12).sort_values("importance")
        plt.barh(top["feature"], top["importance"])
        plt.title("XGBoost Regressor feature importance")
        plt.xlabel("Importance")
        display_and_save("xgboost_regressor_feature_importance.png")

# SVM permutation importance for rain classification
if "SVM" in rain_models:
    svm_model = rain_models["SVM"]
    perm = permutation_importance(
        svm_model,
        X_test,
        y_rain_test.astype(int),
        scoring="f1",
        n_repeats=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    svm_imp = pd.DataFrame({
        "feature": feature_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)
    save_table(svm_imp, "svm_permutation_importance.csv")
    display(svm_imp.head(15))
    plt.figure(figsize=(9, 6))
    top = svm_imp.head(12).sort_values("importance_mean")
    plt.barh(top["feature"], top["importance_mean"])
    plt.title("SVM permutation importance for rain classification")
    plt.xlabel("Mean F1 decrease after permutation")
    display_and_save("svm_permutation_importance.png")


### 11.3 7-day forecast and full-test actual-vs-predicted plots


In [ ]:
forecast20 = seven_iter20["forecast"].copy()
forecast20["forecast_time"] = pd.to_datetime(forecast20["forecast_time"])

plt.figure(figsize=(9, 5))
plt.plot(forecast20["forecast_time"], forecast20["predicted_temperature"], marker="o")
plt.title("Predicted noon temperature for the next 7 days")
plt.xlabel("Forecast date")
plt.ylabel("Predicted temperature (degC)")
plt.xticks(rotation=25, ha="right")
display_and_save("next_7_days_temperature_forecast.png")

plt.figure(figsize=(8, 5))
rain_binary = (forecast20["predicted_rain_label"] == "rain").astype(int)
plt.bar(forecast20["horizon_day"], rain_binary)
plt.title("Predicted rain label for the next 7 days")
plt.xlabel("Forecast horizon day")
plt.ylabel("Rain prediction (1=rain, 0=no rain)")
plt.ylim(0, 1.2)
display_and_save("next_7_days_rain_forecast.png")

test_pred20 = seven_iter20["test_predictions"].copy()
test_pred20["time"] = pd.to_datetime(test_pred20["time"])
horizons = sorted(test_pred20["horizon_day"].unique())

fig, axes = plt.subplots(7, 1, figsize=(13, 22), sharex=False)
for ax, h in zip(axes, horizons):
    part = test_pred20[test_pred20["horizon_day"] == h]
    ax.plot(part["time"], part["actual_temperature"], label="Actual", linewidth=1.1)
    ax.plot(part["time"], part["predicted_temperature"], label="Predicted", linewidth=1.1)
    model = part["temperature_model"].iloc[0]
    ax.set_title(f"Horizon {h}: actual vs predicted temperature ({model})")
    ax.set_ylabel("degC")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Test date")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_actual_vs_predicted_temperature_best_models_by_horizon.png", dpi=160)
plt.show()

fig, axes = plt.subplots(7, 1, figsize=(13, 22), sharex=False)
for ax, h in zip(axes, horizons):
    part = test_pred20[test_pred20["horizon_day"] == h]
    ax.step(part["time"], part["actual_rain"], where="mid", label="Actual rain", linewidth=1.1)
    ax.step(part["time"], part["predicted_rain"], where="mid", label="Predicted rain", linewidth=1.1)
    model = part["rain_model"].iloc[0]
    ax.set_title(f"Horizon {h}: actual vs predicted rain ({model})")
    ax.set_ylabel("Rain label")
    ax.set_ylim(-0.1, 1.1)
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Test date")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_actual_vs_predicted_rain_best_models_by_horizon.png", dpi=160)
plt.show()


### 11.4 Detailed hyperparameter tuning plots by model

This cell re-runs time-series cross-validation searches only to extract `cv_results_` for plotting. It creates the same tuning diagnostics used in the final academic report.

Expected per-model tuning figures:

- `hyperparameter_tuning_temperature_regression_KNNRegressor.png`
- `hyperparameter_tuning_temperature_regression_XGBoostRegressor.png`
- `hyperparameter_tuning_rain_classification_LogisticRegression.png`
- `hyperparameter_tuning_rain_classification_SVM.png`
- `hyperparameter_tuning_rain_classification_XGBoostClassifier.png`
- `per_model_tuning_temperature_cv_rmse_iter4_vs_iter20.png`
- `per_model_tuning_rain_cv_f1_iter4_vs_iter20.png`


In [ ]:
def search_space_size(param_distributions):
    if any(not isinstance(v, (list, tuple)) for v in param_distributions.values()):
        return None
    return len(list(ParameterGrid(param_distributions)))


def extract_cv_rows(horizon, task, model_name, fitted_model, search_iter):
    rows = []
    if not hasattr(fitted_model, "cv_results_"):
        return rows
    results = fitted_model.cv_results_
    best_params = json.dumps(fitted_model.best_params_, sort_keys=True)
    for i, params in enumerate(results["params"]):
        score = float(results["mean_test_score"][i])
        rows.append({
            "horizon_day": horizon,
            "task": task,
            "model": model_name,
            "candidate_index": i + 1,
            "rank_test_score": int(results["rank_test_score"][i]),
            "mean_test_score": score,
            "mean_cv_rmse": -score if task == "temperature_regression" else np.nan,
            "mean_cv_f1": score if task == "rain_classification" else np.nan,
            "is_best": int(results["rank_test_score"][i]) == 1,
            "params": json.dumps(params, sort_keys=True),
            "best_params": best_params,
            "search_iter_budget": search_iter,
        })
    return rows


def collect_7day_tuning_rows(search_iter):
    rows = []
    feature_cols_7 = select_feature_columns(feature_df, include_shortwave=True)
    for horizon in range(1, 8):
        hdf = build_horizon_dataset(noon_df, feature_df, horizon)
        tr, _, _ = chronological_split(hdf)
        Xtr = tr[feature_cols_7]
        ytr_temp = tr["horizon_temperature"]
        ytr_rain = tr["horizon_rain"].astype(int)
        cv = TimeSeriesSplit(n_splits=5)

        temp_models_to_search = {
            "KNNRegressor": (
                Pipeline([("scaler", StandardScaler()), ("model", KNeighborsRegressor())]),
                {"model__n_neighbors": randint(3, 26), "model__weights": ["uniform", "distance"]},
            )
        }
        if XGBRegressor is not None:
            temp_models_to_search["XGBoostRegressor"] = (
                XGBRegressor(objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=1, verbosity=0),
                {"n_estimators": randint(100, 501), "max_depth": randint(3, 11), "learning_rate": uniform(0.01, 0.29)},
            )
        for name, (model, space) in temp_models_to_search.items():
            search = RandomizedSearchCV(model, space, n_iter=search_iter, cv=cv, scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE, n_jobs=1)
            search.fit(Xtr, ytr_temp)
            rows.extend(extract_cv_rows(horizon, "temperature_regression", name, search, search_iter))

        rain_models_to_search = {
            "LogisticRegression": (
                Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))]),
                {"model__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
            ),
            "SVM": (
                Pipeline([("scaler", StandardScaler()), ("model", SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE))]),
                {"model__C": [0.1, 1.0, 10.0, 100.0], "model__kernel": ["linear", "rbf"]},
            ),
        }
        if XGBClassifier is not None:
            neg, pos = max(1, (ytr_rain == 0).sum()), max(1, (ytr_rain == 1).sum())
            rain_models_to_search["XGBoostClassifier"] = (
                XGBClassifier(objective="binary:logistic", eval_metric="logloss", scale_pos_weight=neg/pos, random_state=RANDOM_STATE, n_jobs=1, verbosity=0),
                {"n_estimators": randint(100, 501), "max_depth": randint(3, 11), "learning_rate": uniform(0.01, 0.29)},
            )
        for name, (model, space) in rain_models_to_search.items():
            size = search_space_size(space)
            n_iter = min(search_iter, size) if size is not None else search_iter
            search = RandomizedSearchCV(model, space, n_iter=n_iter, cv=cv, scoring="f1", random_state=RANDOM_STATE, n_jobs=1)
            search.fit(Xtr, ytr_rain)
            rows.extend(extract_cv_rows(horizon, "rain_classification", name, search, search_iter))
    return pd.DataFrame(rows)

# This can take a few minutes because it re-runs tuning for diagnostic plots.
tuning_iter4 = collect_7day_tuning_rows(search_iter=4)
tuning_iter20 = collect_7day_tuning_rows(search_iter=20)
tuning_all = pd.concat([tuning_iter4.assign(version="n_iter=4"), tuning_iter20.assign(version="n_iter=20")], ignore_index=True)
save_table(tuning_all, "seven_day_hyperparameter_tuning_audit.csv")
display(tuning_all.head())


In [ ]:
def plot_hyperparameter_tuning_by_model(tuning_df, version_label):
    version_df = tuning_df[tuning_df["version"] == version_label].copy()
    for (task, model), df in version_df.groupby(["task", "model"]):
        metric_col = "mean_cv_rmse" if task == "temperature_regression" else "mean_cv_f1"
        ylabel = "Mean CV RMSE (lower is better)" if task == "temperature_regression" else "Mean CV F1 (higher is better)"
        fig, axes = plt.subplots(7, 1, figsize=(13, 24), sharex=False)
        for ax, h in zip(axes, range(1, 8)):
            part = df[df["horizon_day"] == h].sort_values("candidate_index")
            if part.empty:
                ax.set_visible(False)
                continue
            ax.plot(part["candidate_index"], part[metric_col], marker="o")
            best = part.sort_values("rank_test_score").iloc[0]
            ax.scatter([best["candidate_index"]], [best[metric_col]], color="red", zorder=3)
            ax.set_title(f"{version_label} - Horizon {h}: {model} best params {best['best_params'][:100]}", fontsize=8)
            ax.set_ylabel(ylabel)
            ax.grid(alpha=0.25)
        axes[-1].set_xlabel("Candidate index")
        plt.tight_layout()
        safe_name = f"hyperparameter_tuning_{task}_{model}.png" if version_label == "n_iter=20" else f"hyperparameter_tuning_{task}_{model}_{version_label.replace('=', '')}.png"
        plt.savefig(FIGURES_DIR / safe_name, dpi=160)
        plt.show()

plot_hyperparameter_tuning_by_model(tuning_all, "n_iter=20")

best_cv = tuning_all[tuning_all["is_best"]].copy()
for task, metric_col, ylabel, filename in [
    ("temperature_regression", "mean_cv_rmse", "Best mean CV RMSE", "per_model_tuning_temperature_cv_rmse_iter4_vs_iter20.png"),
    ("rain_classification", "mean_cv_f1", "Best mean CV F1", "per_model_tuning_rain_cv_f1_iter4_vs_iter20.png"),
]:
    models = sorted(best_cv[best_cv["task"] == task]["model"].unique())
    fig, axes = plt.subplots(len(models), 1, figsize=(11, 4 * len(models)), sharex=True)
    if len(models) == 1:
        axes = [axes]
    for ax, model in zip(axes, models):
        for version in ["n_iter=4", "n_iter=20"]:
            part = best_cv[(best_cv["task"] == task) & (best_cv["model"] == model) & (best_cv["version"] == version)].sort_values("horizon_day")
            if not part.empty:
                ax.plot(part["horizon_day"], part[metric_col], marker="o", label=version)
        ax.set_title(f"{model}: best CV score by horizon")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.25)
        ax.legend()
    axes[-1].set_xlabel("Forecast horizon")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=160)
    plt.show()


## 12. Academic report generation


In [ ]:
def df_to_md(df, digits=3):
    d = df.copy()
    for col in d.select_dtypes(include="number").columns:
        d[col] = d[col].round(digits)
    return d.to_markdown(index=False)

summary_rows = pd.DataFrame([
    {"Metric": "Next-day temperature best test RMSE", "Value": temp_test_df[temp_test_df["model"] == best_temp_model]["rmse"].iloc[0]},
    {"Metric": "Next-day temperature best test R2", "Value": temp_test_df[temp_test_df["model"] == best_temp_model]["r2"].iloc[0]},
    {"Metric": "Next-day selected rain test F1", "Value": rain_test_df[rain_test_df["model"] == best_rain_model]["f1"].iloc[0]},
    {"Metric": "7-day mean temperature RMSE n_iter=4", "Value": seven_iter4["selected"]["test_rmse"].mean()},
    {"Metric": "7-day mean temperature RMSE n_iter=20", "Value": seven_iter20["selected"]["test_rmse"].mean()},
    {"Metric": "7-day mean rain F1 n_iter=4", "Value": seven_iter4["selected"]["test_f1"].mean()},
    {"Metric": "7-day mean rain F1 n_iter=20", "Value": seven_iter20["selected"]["test_f1"].mean()},
])

report = f"""# Final Academic Report - Hanoi Noon Weather Forecasting

## Abstract

This notebook builds a complete machine-learning pipeline for Hanoi noon weather prediction. The system solves next-day temperature regression, next-day rain classification, and direct 7-day multi-horizon forecasting. Data are split chronologically into training years 2020-2023, validation year 2024, and test year 2025.

## Dataset

- Raw records: {len(raw_df):,}
- Noon records: {len(noon_df):,}
- Supervised samples: {len(feature_df):,}
- Date range: {noon_df['time'].min()} to {noon_df['time'].max()}

## Summary metrics

{df_to_md(summary_rows)}

## Next-day temperature regression

Validation results:

{df_to_md(temp_val_df)}

Test results:

{df_to_md(temp_test_df)}

The selected next-day temperature model is **{best_temp_model}**. Linear Regression is trained by estimating the coefficient vector `w` and intercept `b` through least squares minimization, not by hyperparameter tuning.

## Next-day rain classification

Validation results:

{df_to_md(rain_val_df)}

Test results:

{df_to_md(rain_test_df)}

The selected next-day rain model is **{best_rain_model}** according to validation F1.

## Seven-day forecasting

Selected models for `n_iter=20`:

{df_to_md(seven_iter20['selected'])}

Forecast output for the next 7 days:

{df_to_md(seven_iter20['forecast'])}

## n_iter=4 vs n_iter=20 comparison

{df_to_md(comparison)}

The tuned `n_iter=20` version improves average 7-day temperature RMSE and slightly improves average rain F1. The improvement is stronger for longer temperature horizons, while rain prediction remains difficult because rain is sparse and highly variable.

## Additional figures generated in this notebook

The notebook also generates the following final-report figures:

- A/B shortwave-radiation comparison.
- XGBoost Regressor feature importance.
- Linear Regression coefficients `w`.
- SVM permutation importance.
- Per-model hyperparameter tuning plots for KNN, XGBoost, Logistic Regression, SVM, and XGBoost Classifier.
- Full-test actual-vs-predicted temperature and rain plots for 7-day forecasting.
- Next-7-day temperature and rain forecast plots.

## Discussion

Temperature forecasting is easier than rain classification because temperature has stronger temporal continuity and seasonal structure. Rain is sparse, local, and highly variable, so threshold tuning improves the decision rule but cannot fully solve the information limitation of using one historical station-like time series.

## Limitations

The dataset uses one location and only the 12:00 timestamp. The 7-day model predicts future horizons directly from historical features rather than from true future meteorological covariates. A production system should incorporate numerical weather prediction variables, spatial observations, or radar/cloud imagery.

## Conclusion

The project demonstrates that traditional machine-learning models can produce interpretable weather prediction baselines. Linear Regression is strong for next-day temperature, while XGBoost becomes more useful in the 7-day setting. Rain classification benefits from threshold tuning, but further improvement would require richer future meteorological covariates or spatial weather information.
"""

report_path = REPORTS_DIR / "final_academic_report_from_notebook.md"
report_path.write_text(report, encoding="utf-8")
print("Saved", report_path)
print(report[:3000])


## 13. Download generated outputs


In [ ]:
# Create a zip archive containing all generated reports, figures, and tables.
!zip -qr reports_notebook_outputs.zip reports_notebook
!ls -lh reports_notebook_outputs.zip
try:
    from google.colab import files
    files.download('reports_notebook_outputs.zip')
except Exception as exc:
    print('Not running in Colab or download unavailable. File is saved as reports_notebook_outputs.zip')
    print(exc)
